# 📚 Importing Required Libraries & Loading the Dataset

This section imports all essential Python libraries used for data cleaning, inspection, and preparation.  
We then load the merged monthly dataset .

## 🔧 Libraries Used
- **pandas** for data manipulation  
- **numpy** for numerical operations  

## 📥 Loading the Dataset
The dataset `merged_monthly_dataset.csv` contains the merged macroeconomic indicators for UK inflation forecasting.


In [6]:
# Importing necessary libraries
import pandas as pd
import numpy as np
# Display settings
pd.set_option('display.max_columns', None)

# Load the merged dataset
df = pd.read_csv("merged_monthly_dataset.csv")

# Preview first few rows
df.head()


,CPI ANNUAL RATE 00: ALL ITEMS 2015=100,Date,YearMonth,Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA,Bank Rate,Exchange_USD,"Unemployment rate (aged 16 and over, seasonally adjusted): %",RPI: Percentage change over 12 months - Petrol and Oil incl Fuel Oil
0,NaN,1971-02-01,1971-02,NaN,NaN,NaN,3.8,NaN
1,NaN,1971-03-01,1971-03,NaN,NaN,NaN,3.9,NaN
2,NaN,1971-04-01,1971-04,NaN,NaN,NaN,4.0,NaN
3,NaN,1971-05-01,1971-05,NaN,NaN,NaN,4.1,NaN
4,NaN,1971-06-01,1971-06,NaN,NaN,NaN,4.1,NaN


# 🔍 Dataset Structure Overview

In this step, we inspect the structure of the cleaned merged dataset.  
The goal is to understand:

- Total number of rows and columns  
- Column names  
- Data types of each column  
- Memory usage  
- Overall structure of the dataset  

This helps identify data type issues, missing values, and prepares the dataset for further cleaning and feature engineering.


In [5]:
# Display the shape of the dataset (rows, columns)
print("Shape of dataset:", df.shape)

# Display column names
print("\nColumn Names:")
print(df.columns.tolist())

# Show data types and non-null counts
print("\nDataset Info:")
df.info()

# Optional: View summary of each column's data type
print("\nData Types:")
print(df.dtypes)


Shape of dataset: (658, 8)

Column Names:
['CPI ANNUAL RATE 00: ALL ITEMS 2015=100', 'Date', 'YearMonth', 'Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA', 'Bank Rate', 'Exchange_USD', 'Unemployment rate (aged 16 and over, seasonally adjusted): %', 'RPI: Percentage change over 12 months - Petrol and Oil incl Fuel Oil']

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 658 entries, 0 to 657
Data columns (total 8 columns):
 #   Column                                                                Non-Null Count  Dtype  
---  ------                                                                --------------  -----  
 0   CPI ANNUAL RATE 00: ALL ITEMS 2015=100                                441 non-null    float64
 1   Date                                                                  658 non-null    object 
 2   YearMonth                                                             658 non-null    object 
 3   Gross Value Added - Monthly (3 month on 3 m

# 🗓️ Step 1: Convert Date Columns to Proper Datetime Format

In this step, we convert the `Date`  columns from object type to proper `datetime` format.

### Why this matters?
- Ensures correct time-series ordering  
- Enables resampling, merging, lag creation, and plotting  
- Prevents future modeling errors caused by string dates  

We use `pd.to_datetime()` with safe parsing to avoid errors.


In [7]:
# Convert Date column to datetime format
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
# Confirm conversion
print(df[['Date']].dtypes)
df.head()


Date    datetime64[ns]
dtype: object


,CPI ANNUAL RATE 00: ALL ITEMS 2015=100,Date,YearMonth,Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA,Bank Rate,Exchange_USD,"Unemployment rate (aged 16 and over, seasonally adjusted): %",RPI: Percentage change over 12 months - Petrol and Oil incl Fuel Oil
0,NaN,1971-02-01,1971-02,NaN,NaN,NaN,3.8,NaN
1,NaN,1971-03-01,1971-03,NaN,NaN,NaN,3.9,NaN
2,NaN,1971-04-01,1971-04,NaN,NaN,NaN,4.0,NaN
3,NaN,1971-05-01,1971-05,NaN,NaN,NaN,4.1,NaN
4,NaN,1971-06-01,1971-06,NaN,NaN,NaN,4.1,NaN


In [8]:
df.tail(10)

,CPI ANNUAL RATE 00: ALL ITEMS 2015=100,Date,YearMonth,Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA,Bank Rate,Exchange_USD,"Unemployment rate (aged 16 and over, seasonally adjusted): %",RPI: Percentage change over 12 months - Petrol and Oil incl Fuel Oil
648,2.8,2025-02-01,2025-02,0.6,4.50,1.2545,4.5,-2.4
649,2.6,2025-03-01,2025-03,0.7,4.50,1.2911,4.6,-5.4
650,3.5,2025-04-01,2025-04,0.7,4.50,1.3131,4.7,-9.8
651,3.4,2025-05-01,2025-05,0.5,4.25,1.3366,4.7,-11.5
652,3.6,2025-06-01,2025-06,0.3,4.25,1.3566,4.7,-10.0
653,3.8,2025-07-01,2025-07,0.2,4.25,1.3492,4.8,-7.0
654,3.8,2025-08-01,2025-08,0.3,4.00,1.3450,NaN,-5.2
655,3.8,2025-09-01,2025-09,NaN,4.00,NaN,NaN,-1.4
656,NaN,2025-10-01,2025-10,NaN,4.00,NaN,NaN,1.1
657,NaN,2025-11-01,2025-11,NaN,4.00,NaN,NaN,NaN


# 📅 Start and End Valid Dates for Each Column

In this step, we identify the **first** and **last** dates where each data column  
(excluding `Date` and `YearMonth`) contains a non-missing value.

This helps us understand:

- When each variable begins historically  
- When each variable ends  
- Why some variables cause missing rows in early years  

No rows are dropped — this is only for inspection.


In [10]:
# Select columns except Date and YearMonth
value_columns = df.columns.drop(['Date', 'YearMonth'])

value_columns


Index(['CPI ANNUAL RATE 00: ALL ITEMS 2015=100',
       'Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA',
       'Bank Rate', 'Exchange_USD',
       'Unemployment rate (aged 16 and over, seasonally adjusted): %',
       'RPI: Percentage change over 12 months - Petrol and Oil incl Fuel Oil'],
      dtype='object')

In [11]:
# Dictionary to store results
start_end_dates = {}

# Loop through each column except Date and YearMonth
for col in value_columns:
    # Find non-missing rows for this column
    non_missing = df[df[col].notna()]
    
    # Get earliest and latest date for non-missing values
    start_date = non_missing['Date'].min()
    end_date = non_missing['Date'].max()
    
    # Store in dictionary
    start_end_dates[col] = {
        'First Valid Date': start_date,
        'Last Valid Date': end_date
    }

# Convert results to a DataFrame for clean display
start_end_dates_df = pd.DataFrame(start_end_dates).T

start_end_dates_df


,First Valid Date,Last Valid Date
CPI ANNUAL RATE 00: ALL ITEMS 2015=100,1989-01-01,2025-09-01
Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA,1997-06-01,2025-08-01
Bank Rate,2015-11-01,2025-11-01
Exchange_USD,1997-01-01,2025-08-01
"Unemployment rate (aged 16 and over, seasonally adjusted): %",1971-02-01,2025-07-01
RPI: Percentage change over 12 months - Petrol and Oil incl Fuel Oil,1988-01-01,2025-10-01


# 📆 Selecting Rows for the Desired Date Range (1997-06-01 to 2025-07-01)

In this step, we extract only the rows between the dates:

**1997-06-01 → 2025-07-01**

This gives us the longest continuous period where all variables *except Bank Rate*  
have complete historical coverage.



In [12]:
# Define the date range
start_date = pd.to_datetime("1997-06-01")
end_date = pd.to_datetime("2025-07-01")

# Select rows within the date range (without removing columns)
df_selected = df[(df['Date'] >= start_date) & (df['Date'] <= end_date)]

# Display the shape and preview
print("Selected rows shape:", df_selected.shape)
df_selected.head()


Selected rows shape: (338, 8)


,CPI ANNUAL RATE 00: ALL ITEMS 2015=100,Date,YearMonth,Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA,Bank Rate,Exchange_USD,"Unemployment rate (aged 16 and over, seasonally adjusted): %",RPI: Percentage change over 12 months - Petrol and Oil incl Fuel Oil
316,1.7,1997-06-01,1997-06,1.0,NaN,1.6446,7.3,9.3
317,2.0,1997-07-01,1997-07,0.6,NaN,1.6702,7.1,14.0
318,2.0,1997-08-01,1997-08,0.9,NaN,1.6034,6.8,14.1
319,1.8,1997-09-01,1997-09,0.8,NaN,1.6015,6.7,11.2
320,1.9,1997-10-01,1997-10,1.0,NaN,1.6329,6.6,8.5


## 🔍 Checking Missing Values in the Selected Date Range (Excluding Bank Rate)

Now that we have filtered the dataset to the period **1997-06-01 to 2025-07-01**,  
we check for missing values **only in the other variables**, ignoring `Bank Rate`.

### Why exclude Bank Rate?
- Bank Rate starts very late (2015), but we are using a longer modelling window.
- We only want to inspect missing patterns for the remaining variables.

### What we compute:
- Missing values per column (excluding Bank Rate)
- Number of rows with missing values (excluding Bank Rate)
- Percentage of missing values per column


In [13]:
# Columns to check (exclude Bank Rate)
cols_excluding_bankrate = df_selected.columns.drop("Bank Rate")

# 1. Missing values per column
missing_counts = df_selected[cols_excluding_bankrate].isna().sum()

# 2. Percentage missing per column
missing_percent = (df_selected[cols_excluding_bankrate].isna().mean() * 100).round(2)

# 3. Number of rows with ANY missing values (excluding Bank Rate)
rows_with_missing = df_selected[cols_excluding_bankrate].isna().any(axis=1).sum()

print("🔹 Missing Values Per Column (Excluding Bank Rate):\n")
print(missing_counts)

print("\n🔹 Missing Percentage Per Column (%):\n")
print(missing_percent)

print("\n🔹 Total Rows with at Least One Missing Value (Excluding Bank Rate):", rows_with_missing)
print("🔹 Total Rows:", df_selected.shape[0])


🔹 Missing Values Per Column (Excluding Bank Rate):

CPI ANNUAL RATE 00: ALL ITEMS 2015=100                                  0
Date                                                                    0
YearMonth                                                               0
Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA         0
Exchange_USD                                                            0
Unemployment rate (aged 16 and over, seasonally adjusted): %            0
RPI: Percentage change over 12 months - Petrol and Oil incl Fuel Oil    0
dtype: int64

🔹 Missing Percentage Per Column (%):

CPI ANNUAL RATE 00: ALL ITEMS 2015=100                                  0.0
Date                                                                    0.0
YearMonth                                                               0.0
Gross Value Added - Monthly (3 month on 3 month growth) :CVM SA         0.0
Exchange_USD                                                            0.

# 💾 Saving the Cleaned Merged Dataset

Now that we have selected the usable modelling period  
(**1997-06-01 → 2025-07-01**), we save this cleaned dataset as:

**clean_merged_dataset.csv**

⚠️ No columns are removed; the file simply contains the selected rows.


In [15]:
# Save df_selected to the working directory
df_selected.to_csv("clean_merged_dataset.csv", index=False)

print("File saved successfully as: clean_merged_dataset.csv")


File saved successfully as: clean_merged_dataset.csv
